# Analog Pre-Holidays

Forecasts the `PREVIOUSLY_W_HOURS` window immediately before each listed holiday using **`AnalogSpecialDays`** (`analog/analog_special_days.py`).

Unlike the classic `AnalogKNN`, candidate X/X2 pairs are restricted to historical windows whose X2 block overlaps with a pre-holiday period — i.e., the model only learns from past instances where the series was approaching a holiday.

Pipeline:
1. Optionally tune the analog hyperparameters with Optuna over historical holiday windows (`tune_analog_pre_holidays_optuna`).
2. Run `run_analog_pre_holidays_batch` over every (target_date, unique_id) pair.
3. A copy of `holiday_demand_mx.csv` named `pre_holiday_demand_mx_AAAA_MM_DD_HH_MM.csv` is written with the rounded integer forecasts overwriting the pre-holiday rows.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import analog_holidays.analog.P_analog_pre_holidays as pre_holidays_module
import analog_holidays.analog.analog_special_days as analog_special_days_module
import analog_holidays.analog.analog_holidays as analog_holidays_module

pre_holidays_module        = importlib.reload(pre_holidays_module)
analog_special_days_module = importlib.reload(analog_special_days_module)
analog_holidays_module     = importlib.reload(analog_holidays_module)

from analog_holidays.analog.P_analog_pre_holidays import (
    # Stage 1 — pre-holiday window
    list_unique_ids,
    load_pre_holiday_source,
    plot_pre_holiday_batch_grid,
    plot_pre_holiday_run,
    run_analog_pre_holidays_batch,
    tune_analog_pre_holidays_optuna,
    # Stage 2 — holiday day (24 h)
    plot_holiday_day_batch_grid,
    plot_holiday_day_run,
    run_holiday_day_batch,
)

from analog_holidays.analog.analog_holidays import (
    plot_batch_inference_grid,
    plot_batch_pair_sequences_grid,
    run_analog_holidays_batch,
    tune_analog_holidays_optuna,   # used in Stage 3 Optuna tuning
)

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)


## Parameters

- `SOURCE_PATH`: hourly wide CSV with one column per series plus `<uid>_holiday` flags.
- `PREVIOUSLY_W_HOURS`: how many hours immediately before each holiday's `00:00` are forecast.
- `MIN_SPECIAL_POINTS`: minimum number of pre-holiday hours required inside a 24-h X2 candidate block. `None` → defaults to `PREVIOUSLY_W_HOURS` (require the full pre-holiday window).
- `MIN_EVENT_GAP`: minimum separation (hours) between two selected historical events. `None` → defaults to `SEASON_LENGTH`.
- `MAX_EVENTS`: cap on how many historical pre-holiday events are used. `None` = all available.
- `UNIQUE_IDS`: subset of series to forecast (set to `None` to use every series found in the CSV).
- `TUNE_UNIQUE_ID`: series used by the Optuna tuner (one regional study is enough).


In [ ]:
SOURCE_PATH = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_demand_mx.csv'
SELECTOR_FEATURES_PATH = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_selector_features.csv'
CLUSTER_COLUMN = 'analog_cluster'
MATCH_TARGET_CLUSTER = True

if SOURCE_PATH.name != 'holiday_demand_mx.csv':
    raise ValueError(
        'Historical demand source must be holiday_demand_mx.csv. '
        'Do not use holiday_demand_mx_analog_cluster.csv as SOURCE_PATH.'
    )

PREVIOUSLY_W_HOURS = 14
SEASON_LENGTH = PREVIOUSLY_W_HOURS  # debe ser igual a PREVIOUSLY_W_HOURS para que X2 = ventana pre-festiva exacta
K = 10
MIN_SPECIAL_POINTS = None  # None → defaults to PREVIOUSLY_W_HOURS dentro del modelo
MIN_EVENT_GAP = None       # None → defaults to SEASON_LENGTH
MAX_EVENTS = None
TYPEDIST = 'pearson'
TYPEREG = 'PCR'
N_COMPONENTS = 3

UNIQUE_IDS = None  # None = every value column in the CSV
TUNE_UNIQUE_ID = 'SEN_demand_SIN'

DATE_END = '2024-01-01'
OPTUNA_N_TRIALS = 25
OPTUNA_TIMEOUT_SEC = 900
OPTUNA_MAX_EVAL_DATES = 12
OPTUNA_RANDOM_SEED = 42


In [ ]:
TARGET_DATES_2025= [
    ('2025-01-01', "New Year's Day"),
    ('2025-02-03', 'Constitution Day'),
    ('2025-03-17', "Benito Juarez's Birthday"),
    ('2025-04-17', 'Maundy Thursday'),
    ('2025-04-18', 'Good Friday'),
    ('2025-04-19', 'Holy Saturday'),
    ('2025-05-01', 'Labor Day'),
    ('2025-09-16', 'Independence Day'),
    ('2025-11-17', 'Mexican Revolution Day'),
    ('2025-12-24', 'Christmas Eve'),
    ('2025-12-25', 'Christmas Day'),
    ('2025-12-31', "New Year's Eve"),
    # ===== 2026 =====
    ('2026-01-01', "New Year's Day"),
    ('2026-02-02', 'Constitution Day'),
    ('2026-03-16', "Benito Juarez's Birthday"),
    ('2026-04-02', 'Maundy Thursday'),
    ('2026-04-03', 'Good Friday'),
    ('2026-04-04', 'Holy Saturday'),
    ('2026-05-01', 'Labor Day'),
]

TARGET_DATE = TARGET_DATES_2025[-1][0]

## Available series

Inspect the unique IDs present in the source CSV. Set `UNIQUE_IDS` to a sublist above if only some series should be forecast.

In [ ]:
df_source = load_pre_holiday_source(SOURCE_PATH)
available_ids = list_unique_ids(df_source)
print('Series in source:', available_ids)
df_source.head()

## Optuna tuning (optional)

Tunes `AnalogKNN` hyperparameters back-testing on historical holidays for the chosen series.

In [ ]:
# Tune per-series — stores best config independently for each unique_id
optuna_s1_configs = {}
_tune_ids = list_unique_ids(df_source) if UNIQUE_IDS is None else list(UNIQUE_IDS)

for uid in _tune_ids:
    print(f'\n── Optuna Etapa 1 | {uid} ──')
    _opt = tune_analog_pre_holidays_optuna(
        unique_id=uid,
        source_path=SOURCE_PATH,
        train_end=DATE_END,
        previously_w_hours=PREVIOUSLY_W_HOURS,
        season_length=SEASON_LENGTH,
        initial_n_components=N_COMPONENTS,
        n_trials=OPTUNA_N_TRIALS,
        timeout_sec=OPTUNA_TIMEOUT_SEC,
        max_eval_dates=OPTUNA_MAX_EVAL_DATES,
        random_seed=OPTUNA_RANDOM_SEED,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
    )
    optuna_s1_configs[uid] = _opt.best_config
    display(_opt.summary_df)

print('\nMejores hiperparámetros por serie — Etapa 1:')
for uid, cfg in optuna_s1_configs.items():
    print(f'  {uid}: k={int(cfg["k"])}  dist={cfg["typedist"]}  reg={cfg["typereg"]}  nc={int(cfg["n_components"])}')


## Batch forecast + CSV export

Writes `pre_holiday_demand_mx_AAAA_MM_DD_HH_MM.csv` next to the source CSV with the integer-rounded forecasts overwriting the pre-holiday rows.

In [ ]:
from datetime import datetime
from types import SimpleNamespace

_uids_s1 = list_unique_ids(df_source) if UNIQUE_IDS is None else list(UNIQUE_IDS)
df_out_s1 = df_source.copy()
_runs_s1 = {}
_rows_s1 = []

for uid in _uids_s1:
    _def_s1 = {'k': K, 'typedist': TYPEDIST, 'typereg': TYPEREG, 'n_components': N_COMPONENTS}
    cfg_s1 = optuna_s1_configs.get(uid, _def_s1) if 'optuna_s1_configs' in dir() else _def_s1
    print(f'{uid}: k={int(cfg_s1["k"])}  dist={cfg_s1["typedist"]}  reg={cfg_s1["typereg"]}  nc={int(cfg_s1["n_components"])}')
    _part = run_analog_pre_holidays_batch(
        target_dates=TARGET_DATES_2025,
        source_path=SOURCE_PATH,
        unique_ids=[uid],
        previously_w_hours=PREVIOUSLY_W_HOURS,
        season_length=SEASON_LENGTH,
        k=int(cfg_s1['k']),
        typedist=cfg_s1['typedist'],
        typereg=cfg_s1['typereg'],
        n_components=int(cfg_s1['n_components']),
        min_special_points=MIN_SPECIAL_POINTS,
        min_event_gap=MIN_EVENT_GAP,
        max_events=MAX_EVENTS,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
        write_csv=False,
    )
    _runs_s1.update(_part.runs)
    _rows_s1.append(_part.results_df)
    for (dk, u), ro in _part.runs.items():
        if u != uid or ro.fail:
            continue
        ps = ro.pre_holiday_start
        pw = ro.previously_w_hours
        m = (df_out_s1['ds'] >= ps) & (df_out_s1['ds'] < ps + pd.Timedelta(hours=pw))
        nm = int(m.sum())
        if nm > 0:
            df_out_s1.loc[m, uid] = ro.forecast_int[:nm]

_ts = datetime.now().strftime('%Y_%m_%d_%H_%M')
_out_s1 = Path(SOURCE_PATH).parent / f'pre_holiday_demand_mx_{_ts}.csv'
df_out_s1.to_csv(_out_s1, index=False)

batch_result = SimpleNamespace(
    runs=_runs_s1,
    results_df=pd.concat(_rows_s1, ignore_index=True),
    output_path=_out_s1,
    source_path=Path(SOURCE_PATH),
    config={
        'previously_w_hours': PREVIOUSLY_W_HOURS,
        'unique_ids': _uids_s1,
        'match_target_cluster': MATCH_TARGET_CLUSTER,
        'cluster_column': CLUSTER_COLUMN,
        'selector_features_path': SELECTOR_FEATURES_PATH,
    },
)

print(f'\nOutput: {_out_s1.name}')
display(batch_result.results_df.head(20))
batch_result.results_df.groupby('unique_id')[['mae', 'mape_pct']].mean()


## Grid of forecasts for one series

In [ ]:
grid_uid = (UNIQUE_IDS or available_ids)[0]
_gc_s1 = optuna_s1_configs.get(grid_uid, {'k': K, 'typedist': TYPEDIST, 'typereg': TYPEREG}) if 'optuna_s1_configs' in dir() else {'k': K, 'typedist': TYPEDIST, 'typereg': TYPEREG}
fig, axes = plot_pre_holiday_batch_grid(
    batch_result,
    unique_id=grid_uid,
    title=f'Pre-holiday forecasts | {grid_uid} | {_gc_s1["typereg"]} | {_gc_s1["typedist"]} | k={int(_gc_s1["k"])}',
)
plt.show()


## Verify the exported CSV

Reload the new file and compare against the source on the pre-holiday window.

In [ ]:
df_out = pd.read_csv(batch_result.output_path, parse_dates=['ds'])
uid_check = (UNIQUE_IDS or available_ids)[0]
target_ts = pd.Timestamp(TARGET_DATE).normalize()
pre_start = target_ts - pd.Timedelta(hours=PREVIOUSLY_W_HOURS)
mask = (df_out['ds'] >= pre_start) & (df_out['ds'] < target_ts)
df_compare = pd.DataFrame({
    'ds': df_out.loc[mask, 'ds'].to_numpy(),
    'source': df_source.loc[mask, uid_check].to_numpy(),
    'overwritten': df_out.loc[mask, uid_check].to_numpy(),
})
df_compare

---

## Etapa 2 — Pronóstico del día festivo completo (24 h)

Usa el CSV curado de Etapa 1 (`pre_holiday_demand_mx_*.csv`) como fuente — las horas pre-festivas ya están rellenas — y aplica **`AnalogSpecialDays`** con `season_length=24` para pronosticar las 24 h de cada día festivo.

La salida se escribe en `holiday_demand_mx_complete_YYYY_MM_DD_HH_MM.csv`:

| Horas | Origen |
|-------|--------|
| Históricas (antes de `target − 14 h`) | CSV original sin modificar |
| Pre-festivas `[target − 14 h, target)` | Etapa 1 (`AnalogSpecialDays`, `season_length=14`) |
| Día festivo `[target 00:00, target+24 h)` | **Etapa 2** (`AnalogSpecialDays`, `season_length=24`) |

> El análogo usa `MIN_SPECIAL_POINTS_H = 24` para exigir que el bloque X2 candidate cubra el día festivo completo (igual que `P_analog_holidays.ipynb`).


In [ ]:
# ── Etapa 2 — parámetros ────────────────────────────────────────────
# Fuente: output de Etapa 1 (pre_holiday_demand_mx_*.csv)
# Si el kernel fue reiniciado sin ejecutar Etapa 1, apunta al archivo más reciente.
import glob

try:
    STAGE2_SOURCE = batch_result.output_path
except NameError:
    candidates = sorted(
        glob.glob(str(SOURCE_PATH.parent / 'pre_holiday_demand_mx_*.csv'))
    )
    if not candidates:
        raise FileNotFoundError('No pre_holiday_demand_mx_*.csv found. Run Stage 1 first.')
    STAGE2_SOURCE = Path(candidates[-1])

print(f'Etapa 2 fuente: {STAGE2_SOURCE.name}')

# Hiperparámetros para el pronóstico del día festivo (24 h)
SEASON_LENGTH_H  = 24           # siempre 24 h para cubrir el día completo
K_H              = K            # reutiliza el K optimizado de Etapa 1
MIN_SP_H         = 24           # exige el día festivo completo en X2 (igual que P_analog_holidays)
MIN_GAP_H        = 24
MAX_EVT_H        = None
TYPEDIST_H       = TYPEDIST
TYPEREG_H        = TYPEREG
N_COMPONENTS_H   = N_COMPONENTS
LEVELS_H         = [80, 95]     # intervalos de predicción (igual que P_analog_holidays)


In [ ]:
# Tune per-series for Stage 2 (holiday day, 24 h) using tune_analog_holidays_optuna
optuna_s2_configs = {}
_uids_s2 = batch_result.config['unique_ids']

for uid in _uids_s2:
    print(f'\n── Optuna Etapa 2 | {uid} ──')
    _opt2 = tune_analog_holidays_optuna(
        unique_id=uid,
        source_path=STAGE2_SOURCE,
        train_end=DATE_END,
        season_length=SEASON_LENGTH_H,
        initial_k=K_H,
        initial_typedist=TYPEDIST_H,
        initial_typereg=TYPEREG_H,
        initial_n_components=N_COMPONENTS_H,
        n_trials=OPTUNA_N_TRIALS,
        timeout_sec=OPTUNA_TIMEOUT_SEC,
        max_eval_dates=OPTUNA_MAX_EVAL_DATES,
        random_seed=OPTUNA_RANDOM_SEED,
        special_labels=('holiday',),
        min_special_points=MIN_SP_H,
        min_event_gap=MIN_GAP_H,
        max_events=MAX_EVT_H,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
    )
    optuna_s2_configs[uid] = _opt2.best_config
    display(_opt2.summary_df)

print('\nMejores hiperparámetros por serie — Etapa 2:')
for uid, cfg in optuna_s2_configs.items():
    print(f'  {uid}: k={int(cfg["k"])}  dist={cfg["typedist"]}  reg={cfg["typereg"]}  nc={int(cfg["n_components"])}')


### Batch forecast de días festivos + exportación CSV


In [ ]:
from types import SimpleNamespace

_src_s2 = load_pre_holiday_source(STAGE2_SOURCE)
df_out_s2 = _src_s2.copy()
_runs_s2 = {}
_rows_s2 = []

for uid in _uids_s2:
    _def_s2 = {'k': K_H, 'typedist': TYPEDIST_H, 'typereg': TYPEREG_H, 'n_components': N_COMPONENTS_H}
    cfg_s2 = optuna_s2_configs.get(uid, _def_s2) if 'optuna_s2_configs' in dir() else _def_s2
    print(f'{uid}: k={int(cfg_s2["k"])}  dist={cfg_s2["typedist"]}  reg={cfg_s2["typereg"]}  nc={int(cfg_s2["n_components"])}')
    _part2 = run_holiday_day_batch(
        target_dates=TARGET_DATES_2025,
        source_path=STAGE2_SOURCE,
        unique_ids=[uid],
        season_length=SEASON_LENGTH_H,
        k=int(cfg_s2['k']),
        typedist=cfg_s2['typedist'],
        typereg=cfg_s2['typereg'],
        n_components=int(cfg_s2['n_components']),
        min_special_points=MIN_SP_H,
        min_event_gap=MIN_GAP_H,
        max_events=MAX_EVT_H,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
        write_csv=False,
    )
    _runs_s2.update(_part2.runs)
    _rows_s2.append(_part2.results_df)
    for (dk, u), ro in _part2.runs.items():
        if u != uid or ro.fail:
            continue
        d_start = ro.target_date
        d_end = d_start + pd.Timedelta(hours=SEASON_LENGTH_H)
        m = (df_out_s2['ds'] >= d_start) & (df_out_s2['ds'] < d_end)
        nm = int(m.sum())
        if nm > 0:
            df_out_s2.loc[m, uid] = ro.forecast_int[:nm]

from datetime import datetime
_ts2 = datetime.now().strftime('%Y_%m_%d_%H_%M')
_out_s2 = Path(STAGE2_SOURCE).parent / f'holiday_demand_mx_complete_{_ts2}.csv'
df_out_s2.to_csv(_out_s2, index=False)

holiday_batch = SimpleNamespace(
    runs=_runs_s2,
    results_df=pd.concat(_rows_s2, ignore_index=True),
    output_path=_out_s2,
    source_path=Path(STAGE2_SOURCE),
    config={
        'season_length': SEASON_LENGTH_H,
        'unique_ids': _uids_s2,
        'match_target_cluster': MATCH_TARGET_CLUSTER,
        'cluster_column': CLUSTER_COLUMN,
        'selector_features_path': SELECTOR_FEATURES_PATH,
    },
)

print(f'\nEtapa 2 output: {_out_s2.name}')
display(holiday_batch.results_df.head(20))
holiday_batch.results_df.groupby('unique_id')[['mae', 'mape_pct']].mean()


### Tabla resumen por serie — métricas agregadas


In [ ]:
summary = (
    holiday_batch.results_df
    .query('fail == False')
    .groupby(['unique_id', 'target_date', 'holiday_label'])[['mae', 'mape_pct']]
    .first()
    .reset_index()
    .sort_values(['unique_id', 'target_date'])
    .assign(mae=lambda df: df['mae'].round(1),
            mape_pct=lambda df: df['mape_pct'].round(2))
)
display(summary)

# Aggregate by series
(
    summary
    .groupby('unique_id')[['mae', 'mape_pct']]
    .agg(['mean', 'median', 'std'])
    .round(2)
)


### Grid de pronósticos del día festivo — una serie a la vez


In [ ]:
for uid in holiday_batch.config['unique_ids']:
    fig, _ = plot_holiday_day_batch_grid(
        holiday_batch,
        unique_id=uid,
        n_cols=4,
    )
    plt.show()


### Grid de inferencia con intervalos de predicción — estilo P_analog_holidays

Llama a `run_analog_holidays_batch` por serie sobre el CSV curado de Etapa 2 para obtener los intervalos de predicción y graficar con `plot_batch_inference_grid`.


In [ ]:
for uid in holiday_batch.config['unique_ids']:
    _def_v = {'k': K_H, 'typedist': TYPEDIST_H, 'typereg': TYPEREG_H, 'n_components': N_COMPONENTS_H}
    cfg_v = optuna_s2_configs.get(uid, _def_v) if 'optuna_s2_configs' in dir() else _def_v
    batch_r = run_analog_holidays_batch(
        target_dates=TARGET_DATES_2025,
        unique_id=uid,
        source_path=STAGE2_SOURCE,
        season_length=SEASON_LENGTH_H,
        k=int(cfg_v['k']),
        typedist=cfg_v['typedist'],
        typereg=cfg_v['typereg'],
        n_components=int(cfg_v['n_components']),
        levels=LEVELS_H,
        special_labels=('holiday',),
        min_special_points=MIN_SP_H,
        min_event_gap=MIN_GAP_H,
        max_events=MAX_EVT_H,
        expected_target_label=None,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
    )
    fig, axes = plot_batch_inference_grid(
        batch_r,
        title=f'Batch inference | {uid}\n{cfg_v["typereg"]} | {cfg_v["typedist"]} | k={int(cfg_v["k"])}',
    )
    plt.show()


### Pares X/X2 — secuencias análogas por día festivo — estilo P_analog_holidays


In [ ]:
for uid in holiday_batch.config['unique_ids']:
    _def_v = {'k': K_H, 'typedist': TYPEDIST_H, 'typereg': TYPEREG_H, 'n_components': N_COMPONENTS_H}
    cfg_v = optuna_s2_configs.get(uid, _def_v) if 'optuna_s2_configs' in dir() else _def_v
    batch_r = run_analog_holidays_batch(
        target_dates=TARGET_DATES_2025,
        unique_id=uid,
        source_path=STAGE2_SOURCE,
        season_length=SEASON_LENGTH_H,
        k=int(cfg_v['k']),
        typedist=cfg_v['typedist'],
        typereg=cfg_v['typereg'],
        n_components=int(cfg_v['n_components']),
        levels=LEVELS_H,
        special_labels=('holiday',),
        min_special_points=MIN_SP_H,
        min_event_gap=MIN_GAP_H,
        max_events=MAX_EVT_H,
        expected_target_label=None,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
    )
    fig_seq, axes_seq = plot_batch_pair_sequences_grid(batch_r)
    plt.show()


---

## Etapa 3 — Ventana completa: 24 h pre-festivo + 24 h día festivo

Para ver los 48 h completos en los mismos tipos de gráfica de `P_analog_holidays.ipynb`
(`plot_batch_inference_grid` y `plot_batch_pair_sequences_grid`), desplazamos cada
`target_date` 24 h hacia atrás:

```
target ajustado = D − 24 h
predict(h=48)   → [D−24h, D+24h)
                   └─ 24 h pre-festivo ─┘└─ 24 h día festivo ─┘
```

Fuente: `holiday_demand_mx_complete_*.csv` — contiene los datos curados de
Etapa 1 (pre-festivo) y el pronóstico de Etapa 2 (día festivo).

> `season_length=48`, `min_special_points=24` exige que el X2 análogo
> cubra el día festivo completo (24 h marcadas con `*_holiday = 1`).


In [ ]:
import glob

# Fuente: CSV completo (Etapa 1 pre-festivo + Etapa 2 día festivo ya rellenados)
try:
    FULL_SOURCE = holiday_batch.output_path
except NameError:
    _candidates = sorted(
        glob.glob(str(SOURCE_PATH.parent / 'holiday_demand_mx_complete_*.csv'))
    )
    if not _candidates:
        raise FileNotFoundError(
            'No holiday_demand_mx_complete_*.csv encontrado. Ejecuta Etapa 2 primero.'
        )
    FULL_SOURCE = Path(_candidates[-1])

# Desplaza cada target 24 h atrás → forecast cubre [D-24h, D+24h)
TARGET_DATES_FULL = [
    ((pd.Timestamp(d) - pd.Timedelta(hours=24)).strftime('%Y-%m-%d'), lbl)
    for d, lbl in TARGET_DATES_2025
]

SEASON_LENGTH_FULL = 48   # 24 h pre-festivo + 24 h día festivo
MIN_SP_FULL        = 24   # X2 debe contener el día festivo completo
MIN_GAP_FULL       = 48   # separación mínima entre eventos análogos

print(f'Fuente: {FULL_SOURCE.name}')
print(f'\nFechas ajustadas (target = D−24 h):')
for d, lbl in TARGET_DATES_FULL:
    print(f'  {d}  {lbl}')
